# FlashInspector AI – Train on Kaggle (GPU)

Train the fire safety YOLO model on Kaggle's free GPU.

**Before running:**
1. **Settings → Accelerator → GPU** (P100 or T4 x2).
2. Add your [Roboflow API key](https://app.roboflow.com/settings/api) as a **Secret**: Notebook setup → Add-ons → Secrets → add `ROBOFLOW_API_KEY`, or paste it when prompted below.
3. This notebook clones the repo from GitHub; set `REPO_URL` in the next cell if using a fork.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repo and go to project

In [ ]:
REPO_URL = "https://github.com/patrisiyarum/fire.git"

!git clone --depth 1 $REPO_URL /kaggle/working/fire
%cd /kaggle/working/fire/flashinspector-ai
import os
print("Project dir:", os.getcwd())

## 3. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow opencv-python python-dotenv pyyaml tqdm

## 4. Roboflow API key

Uses **Kaggle Secret** `ROBOFLOW_API_KEY` if set (Add-ons → Secrets). Otherwise you'll be prompted to paste it.

In [ ]:
import os

if os.environ.get("ROBOFLOW_API_KEY"):
    print("Using ROBOFLOW_API_KEY from Secrets.")
else:
    from getpass import getpass
    os.environ["ROBOFLOW_API_KEY"] = getpass("Paste your Roboflow API key: ")

## 5. Download all datasets

In [ ]:
!python download_datasets.py

## 6. Merge datasets (multi-class)

In [ ]:
!python prepare_dataset.py

## 7. Train on Kaggle GPU

Trains on the merged dataset (all classes). Output goes to `runs/fire_safety/weights/best.pt`. Colab-friendly: 50 epochs, batch 8.

In [ ]:
!python train.py --epochs 50 --batch 8 --model yolov8n.pt

## 8. Save model to Kaggle Output

Copy `best.pt` to `/kaggle/working/` so it appears in **Output** and you can download it after the run.

In [ ]:
import shutil
from pathlib import Path

src = Path("runs/fire_safety/weights/best.pt")
if src.exists():
    dest = Path("/kaggle/working/best.pt")
    shutil.copy(src, dest)
    print(f"Saved to {dest} — download from the Output tab when the run finishes.")
else:
    print("best.pt not found. Check training output above.")
    !ls -la runs/fire_safety/weights/ 2>/dev/null || true